In [1]:
import torch
import torch.nn as nn
from ultralytics import YOLO

In [29]:
PATH_PT = "../../../models/yolov8n.pt"
PATH_RELU_PT = "../../../models/yolov8n_relu_init.pt"
PATH_DATA = "/home/helbuk/Documents/Thesis_Compiler_Aware_Model_Optimization_2026/thesis2026-project/datasets/coco/data.yaml"

In [20]:
from pathlib import Path
import yaml

print("PATH_DATA =", PATH_DATA)
print("resolved =", Path(PATH_DATA).resolve())
print("exists =", Path(PATH_DATA).exists())

with open(Path(PATH_DATA).resolve(), "r") as f:
    print(f.read())

PATH_DATA = ../../../datasets/coco/data.yaml
resolved = /home/helbuk/Documents/Thesis_Compiler_Aware_Model_Optimization_2026/thesis2026-project/datasets/coco/data.yaml
exists = True
path: .

train: train/images
val: valid/images

task: detect

# Classes
names:
  0: person
  1: bicycle
  2: car
  3: motorcycle
  4: airplane
  5: bus
  6: train
  7: truck
  8: boat
  9: traffic light
  10: fire hydrant
  11: stop sign
  12: parking meter
  13: bench
  14: bird
  15: cat
  16: dog
  17: horse
  18: sheep
  19: cow
  20: elephant
  21: bear
  22: zebra
  23: giraffe
  24: backpack
  25: umbrella
  26: handbag
  27: tie
  28: suitcase
  29: frisbee
  30: skis
  31: snowboard
  32: sports ball
  33: kite
  34: baseball bat
  35: baseball glove
  36: skateboard
  37: surfboard
  38: tennis racket
  39: bottle
  40: wine glass
  41: cup
  42: fork
  43: knife
  44: spoon
  45: bowl
  46: banana
  47: apple
  48: sandwich
  49: orange
  50: broccoli
  51: carrot
  52: hot dog
  53: pizza
  54: 

In [33]:
def replace_silu_with_relu(module: nn.Module, inplace: bool = True) -> nn.Module:
    for name, child in list(module.named_children()):
        if isinstance(child, nn.SiLU):
            setattr(module, name, nn.ReLU(inplace=inplace))
        else:
            replace_silu_with_relu(child, inplace=inplace)
    return module

model = YOLO(PATH_PT)
replace_silu_with_relu(model.model)

num_silu = sum(1 for m in model.model.modules() if isinstance(m, nn.SiLU))
num_relu = sum(1 for m in model.model.modules() if isinstance(m, nn.ReLU))
print("remaining SiLU:", num_silu)
print("ReLU count:", num_relu)

ckpt_path = PATH_RELU_PT
torch.save({"model": model.model}, ckpt_path)

remaining SiLU: 0
ReLU count: 57


In [35]:
baseline = YOLO(PATH_PT)
relu_model = YOLO(PATH_RELU_PT)

print("Baseline:")
baseline.val(data=PATH_DATA)

print("ReLU fine-tuned:")
relu_model.val(data=PATH_DATA)

Baseline:
Ultralytics 8.4.21 🚀 Python-3.10.12 torch-2.8.0 CUDA:0 (Orin, 7620MiB)
YOLOv8n summary (fused): 72 layers, 3,151,904 parameters, 0 gradients, 8.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 45.8±16.3 MB/s, size: 188.5 KB)
val: Scanning /home/helbuk/Documents/Thesis_Compiler_Aware_Model_Optimization_2026/thesis2026-project/datasets/coco/valid/labels.cache... 4952 images, 48 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 5000/5000 295.4Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 1% ──────────── 3/313 1.3s/it 2.7s<6:394

NvMapMemAllocInternalTagged: 1075072515 error 12
NvMapMemHandleAlloc: error 0
NvMapMemAllocInternalTagged: 1075072515 error 12
NvMapMemHandleAlloc: error 0


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 5% ╸─────────── 17/313 3.2it/s 7.5s<1:33

NvMapMemAllocInternalTagged: 1075072515 error 12
NvMapMemHandleAlloc: error 0
NvMapMemAllocInternalTagged: 1075072515 error 12
NvMapMemHandleAlloc: error 0


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 39% ━━━━╸─────── 121/313 3.0it/s 41.2s<1:04

...

NvMapMemAllocInternalTagged: 1075072515 error 12
NvMapMemHandleAlloc: error 0
NvMapMemAllocInternalTagged: 1075072515 error 12
NvMapMemHandleAlloc: error 0


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 72% ━━━━━━━━╸─── 224/313 2.3it/s 1:18<38.1s

NvMapMemAllocInternalTagged: 1075072515 error 12
NvMapMemHandleAlloc: error 0
NvMapMemAllocInternalTagged: 1075072515 error 12
NvMapMemHandleAlloc: error 0
NvMapMemAllocInternalTagged: 1075072515 error 12
NvMapMemHandleAlloc: error 0


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 313/313 2.9it/s 1:48<0.3s
                   all       5000      36335      0.634      0.475      0.521      0.371
                person       2693      10777      0.756      0.672      0.745      0.514
               bicycle        149        314      0.687      0.392      0.456      0.264
                   car        535       1918      0.648      0.514      0.562      0.365
            motorcycle        159        367      0.712       0.58      0.656      0.413
              airplane         97        143      0.814      0.764      0.834      0.653
                   bus        189        283      0.747      0.643      0.739      0.619
                 train        157        190      0.798      0.769      0.834      0.647
                 truck        250        414      0.548      0.399      0.435      0.294
                  boat        121        424      0.583        0.

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77,
       78, 79])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0xfffe635c6bc0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024

In [36]:
relu_model.model

DetectionModel(
  (model): Sequential(
    (0): Conv(
      (conv): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      (act): ReLU(inplace=True)
    )
    (1): Conv(
      (conv): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      (act): ReLU(inplace=True)
    )
    (2): C2f(
      (cv1): Conv(
        (conv): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1))
        (act): ReLU(inplace=True)
      )
      (cv2): Conv(
        (conv): Conv2d(48, 32, kernel_size=(1, 1), stride=(1, 1))
        (act): ReLU(inplace=True)
      )
      (m): ModuleList(
        (0): Bottleneck(
          (cv1): Conv(
            (conv): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
            (act): ReLU(inplace=True)
          )
          (cv2): Conv(
            (conv): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
            (act): ReLU(inplace=True)
          )
        )
      )
    )
    (3): Conv(
      (conv): Conv2d(32

In [37]:
baseline.model

DetectionModel(
  (model): Sequential(
    (0): Conv(
      (conv): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      (act): SiLU(inplace=True)
    )
    (1): Conv(
      (conv): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      (act): SiLU(inplace=True)
    )
    (2): C2f(
      (cv1): Conv(
        (conv): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1))
        (act): SiLU(inplace=True)
      )
      (cv2): Conv(
        (conv): Conv2d(48, 32, kernel_size=(1, 1), stride=(1, 1))
        (act): SiLU(inplace=True)
      )
      (m): ModuleList(
        (0): Bottleneck(
          (cv1): Conv(
            (conv): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
            (act): SiLU(inplace=True)
          )
          (cv2): Conv(
            (conv): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
            (act): SiLU(inplace=True)
          )
        )
      )
    )
    (3): Conv(
      (conv): Conv2d(32